Step 1: Install Required Libraries

In [1]:
%pip install bertopic
%pip install umap-learn
%pip install hdbscan
%pip install sentence-transformers

You should consider upgrading via the '/usr/local/bin/python3.10 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/usr/local/bin/python3.10 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/usr/local/bin/python3.10 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
You should consider upgrading via the '/usr/local/bin/python3.10 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.



Restart kernel after installation.


Step 2: Import Libraries

In [2]:
import pandas as pd
import numpy as np

In [3]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
import matplotlib.pyplot as plt

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 3: Load Patent Topic Dataset
You already created:
patent_topic_dataset.csv
Load:

In [4]:
DATA_PATH = "../data/processed/patent_topic_dataset.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)

(18118, 10)


Check columns:

In [5]:
df.columns

Index(['patent_id', 'title', 'combined_text', 'filing_year', 'country',
       'assignee_en', 'cpc_codes', 'ipc_codes', 'document_length',
       'topic_text'],
      dtype='object')

In [5]:
embeddings = np.load("../data/processed/patent_embeddings.npy")

print(embeddings.shape)

(18118, 384)


As For BERTopic we need only text. So, 

In [6]:
documents = df["topic_text"].astype(str).tolist()

print(len(documents))

18118


In [9]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="mps"
)

embeddings = embedding_model.encode(
    documents,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

Batches: 100%|██████████| 1133/1133 [03:15<00:00,  5.80it/s]


In [10]:
print(embeddings.shape)

(18118, 384)


In [7]:
import numpy as np

np.save(
    "../data/processed/patent_embeddings.npy",
    embeddings
)

Step 6: Configure BERTopic Components

UMAP

In [8]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

HDBSCAN(create clusters)

In [9]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True
)

In [10]:
#assert len(documents) == embeddings.shape[0]

Step 7: Create BERTopic Model

In [11]:
topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)

Step 8: Train BERTopic

In [12]:
print("Number of documents:", len(documents))
print("Embeddings shape:", embeddings.shape)
print("Type of embeddings:", type(embeddings))

Number of documents: 18118
Embeddings shape: (18118, 384)
Type of embeddings: <class 'numpy.ndarray'>


In [13]:
print(df.shape)

print(df["patent_id"].nunique())

print(df["patent_id"].duplicated().sum())

(18118, 10)
18118
0


In [14]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-07-28 17:51:06,165 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-07-28 17:51:37,733 - BERTopic - Dimensionality - Completed ✓
2026-07-28 17:51:37,737 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-07-28 17:51:43,396 - BERTopic - Cluster - Completed ✓
2026-07-28 17:51:43,406 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-07-28 17:51:57,212 - BERTopic - Representation - Completed ✓


In [15]:
# Add discovered topic IDs to the dataframe
df["topic"] = topics

# (Optional) Add topic probabilities if available
# df["topic_probability"] = probs.max(axis=1)

In [16]:
print(df.columns)

print(df[["patent_id", "topic"]].head())

Index(['patent_id', 'title', 'combined_text', 'filing_year', 'country',
       'assignee_en', 'cpc_codes', 'ipc_codes', 'document_length',
       'topic_text', 'topic'],
      dtype='object')
       patent_id  topic
0   CN102708128A      0
1  KR102798691B1      0
2   US11913937B2      0
3    JP7095207B2      2
4   CN108538402A      0


In [17]:
topic_info = topic_model.get_topic_info()


topic_info.head(20)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25,-1_dehydrated_microorganism_messages_the,"[dehydrated, microorganism, messages, the, coa...",[coated dehydrated microorganisms with enhance...
1,0,14179,0_the_of_and_to,"[the, of, and, to, is, in, data, information, ...",[a system for acquiring ultrasound images of i...
2,1,3281,1_of_parts_is_and,"[of, parts, is, and, the, in, to, powder, that...",[a kind of health care kudzuvine root instant ...
3,2,358,2_the_of_information_to,"[the, of, information, to, unit, and, is, in, ...",[blood sugar control system glycemic control s...
4,3,275,3_of_the_in_is,"[of, the, in, is, and, with, for, to, points, ...",[diagnostic technique for acute coronary syndr...


In [18]:
import os

os.makedirs("../results/bertopic", exist_ok=True)

df.to_csv(
    "../results/bertopic/patent_topics.csv",
    index=False
)

print("patent_topics.csv saved successfully!")

patent_topics.csv saved successfully!


In [19]:
# Save 2:Topic Summary
#Get BERTopic topic information:

topic_info = topic_model.get_topic_info()

topic_info.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,25,-1_dehydrated_microorganism_messages_the,"[dehydrated, microorganism, messages, the, coa...",[coated dehydrated microorganisms with enhance...
1,0,14179,0_the_of_and_to,"[the, of, and, to, is, in, data, information, ...",[a system for acquiring ultrasound images of i...
2,1,3281,1_of_parts_is_and,"[of, parts, is, and, the, in, to, powder, that...",[a kind of health care kudzuvine root instant ...
3,2,358,2_the_of_information_to,"[the, of, information, to, unit, and, is, in, ...",[blood sugar control system glycemic control s...
4,3,275,3_of_the_in_is,"[of, the, in, is, and, with, for, to, points, ...",[diagnostic technique for acute coronary syndr...


In [20]:
#Save 3: Topic Probabilities (optional but recommended)
import numpy as np


np.save(
    "../results/bertopic/topic_probabilities.npy", 
    probabilities
)

print("Topic probabilities saved")

Topic probabilities saved


In [21]:
probabilities = np.load(
    "../results/bertopic/topic_probabilities.npy"
)

probabilities.shape

(18118, 4)

In [22]:
#Save 4: Save BERTopic Model
#This is important.
#Without this, every time you open another notebook you need to train BERTopic again.

topic_model.save(
    "../models/bertopic_model"
)

print("BERTopic model saved")

2026-07-28 17:52:28,488 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


BERTopic model saved


In [27]:
#Save 5: Save Topic Visualization Data 
#For Streamlit, save topic frequency:

topic_frequency = topic_model.get_topic_info()

topic_frequency.to_csv(
    "../results/bertopic/topic_frequency.csv",
    index=False
)

In [28]:
df = pd.read_csv(
    "../results/bertopic/patent_topics.csv"
)

In [29]:
topic_info.to_csv(
    "../results/bertopic/topic_summary.csv",
    index=False
)